# Bio-Lab AI — audited Mistral 7B QLoRA

This notebook trains **only a rank-8 LoRA adapter** on a Google Colab GPU. It never downloads the 7B model to the Mac. It supports an owner-reviewed private export and a clearly labeled public-licensed bootstrap mode. It refuses to train when the dataset, grouped splits, task coverage, privacy scan, token lengths, or adapter package fail their gates. The held-out test set is never used for optimization.

**Data rule:** public bootstrap mode downloads only pinned MIT/CC-BY sources and records a source manifest. For owner-reviewed mode, upload only the private admin export and never put it in Git, a public Drive folder, a public Hugging Face repository, or a public Trackio Space.

In [ ]:
# Exact application-library versions; Colab's CUDA-matched torch build is retained.
import subprocess, sys

!nvidia-smi
PINNED_PACKAGES = [
    'transformers==4.57.6', 'tokenizers==0.22.2', 'trl==1.8.0',
    'peft==0.19.1', 'bitsandbytes==0.49.2', 'datasets==4.7.0',
    'accelerate==1.14.0', 'huggingface_hub==0.36.2',
    'safetensors==0.6.2', 'sentencepiece==0.2.1', 'jedi==0.19.2',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', *PINNED_PACKAGES], check=True)

# Check the isolated training stack, not unrelated packages preinstalled by Colab.
# Colab currently bundles Gradio/Diffusers versions whose optional requirements
# conflict with Transformers' Hub bound even though neither is used here.
from importlib.metadata import version
EXPECTED_VERSIONS = dict(spec.rsplit('==', 1) for spec in PINNED_PACKAGES)
actual_versions = {name: version(name) for name in EXPECTED_VERSIONS}
assert actual_versions == EXPECTED_VERSIONS, (actual_versions, EXPECTED_VERSIONS)
for module in ('transformers', 'tokenizers', 'trl', 'peft', 'bitsandbytes', 'datasets', 'accelerate', 'huggingface_hub', 'safetensors', 'sentencepiece'):
    __import__(module)
print('Pinned training dependency stack installed and importable.')

In [ ]:
import contextlib, csv, gc, hashlib, importlib.metadata, json, math, os, pathlib, random, re, statistics, subprocess, sys, time, urllib.request
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import torch
from google.colab import drive, files
from huggingface_hub import HfApi, notebook_login, whoami
from packaging.version import Version

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
BASE_REVISION = '63a8b081895390a26e140280378bc85ec8bce07a'
DATASET_SCHEMA_VERSION = 2
TRAINING_DATA_MODE = 'public_bootstrap'  # or 'human_export'
assert TRAINING_DATA_MODE in {'public_bootstrap', 'human_export'}
ALLOWED_PROVENANCE = {'public_licensed'} if TRAINING_DATA_MODE == 'public_bootstrap' else {'human_corrected'}
PUBLIC_BOOTSTRAP_SCRIPT_URL = 'https://raw.githubusercontent.com/bbiolab123-cell/Integrated/main/Bio-Lab-AI/training/build_public_bootstrap_dataset.py'
PUBLIC_BOOTSTRAP_SCRIPT_SHA256 = '6ad15c1d4bb24ec30c26be4b2df49dff60fba842adceef9617529f1b4ca6d50c'
SEED = 3407
MAX_LENGTH = 2048
MIN_EXAMPLES = 200
MIN_PER_TASK = 10
MIN_HOLDOUT_EXAMPLES = 10
MIN_HOLDOUT_GROUPS = 2
REQUIRED_TASKS = {
    'experiment_analysis', 'data_analysis', 'experiment_chat',
    'experiment_comparison', 'protocol_generation', 'sop_structuring',
    'project_chat', 'project_synthesis', 'general_chat',
}

assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU. Do not run this on the Mac.'
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 2**30
assert gpu_gib >= 14.0, f'At least 14 GiB GPU memory is required; this runtime has {gpu_gib:.1f} GiB.'
assert Version(torch.__version__.split('+')[0]) >= Version('2.4'), f'Colab torch is too old: {torch.__version__}'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True

PACKAGE_VERSIONS = {name: importlib.metadata.version(name) for name in [
    'torch', 'transformers', 'tokenizers', 'trl', 'peft', 'bitsandbytes',
    'datasets', 'accelerate', 'huggingface_hub', 'safetensors',
]}
print(f'GPU: {gpu.name} ({gpu_gib:.1f} GiB)')
print(json.dumps(PACKAGE_VERSIONS, indent=2))

# Drive stores resumable checkpoints and audit artifacts. Raw JSONL remains in the private Colab runtime.
drive.mount('/content/drive')
RUNS_ROOT = pathlib.Path('/content/drive/MyDrive/BioLab-AI-private-training')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Hugging Face login is deferred until release gates pass; training needs no Hub write token.')

## 1. Build or upload and validate the training dataset

The default public-bootstrap mode deterministically builds exactly 200 examples from pinned, reusable sources without calling an LLM. Change `TRAINING_DATA_MODE` to `human_export` to upload `biolab-ai-training.jsonl` from the owner-only admin page. Validation is strict in both modes.

In [ ]:
bootstrap_manifest_path = None
if TRAINING_DATA_MODE == 'public_bootstrap':
    script_bytes = urllib.request.urlopen(PUBLIC_BOOTSTRAP_SCRIPT_URL, timeout=45).read()
    script_sha256 = hashlib.sha256(script_bytes).hexdigest()
    assert script_sha256 == PUBLIC_BOOTSTRAP_SCRIPT_SHA256, 'Public bootstrap builder changed; review and update its pinned hash.'
    script_path = pathlib.Path('/content/build_public_bootstrap_dataset.py')
    script_path.write_bytes(script_bytes)
    dataset_path = '/content/biolab-public-bootstrap.jsonl'
    bootstrap_manifest_path = pathlib.Path('/content/biolab-public-bootstrap-manifest.json')
    subprocess.run([
        sys.executable, str(script_path), '--output', dataset_path,
        '--manifest', str(bootstrap_manifest_path),
    ], check=True)
    dataset_bytes = pathlib.Path(dataset_path).read_bytes()
else:
    uploaded = files.upload()
    jsonl_names = [name for name in uploaded if name.endswith('.jsonl')]
    assert len(jsonl_names) == 1, 'Upload exactly one .jsonl training export.'
    dataset_path = jsonl_names[0]
    dataset_bytes = uploaded[dataset_path]
dataset_sha256 = hashlib.sha256(dataset_bytes).hexdigest()

rows = []
for line_number, raw_line in enumerate(dataset_bytes.decode('utf-8').splitlines(), start=1):
    if not raw_line.strip():
        continue
    try:
        rows.append(json.loads(raw_line))
    except json.JSONDecodeError as exc:
        raise AssertionError(f'Line {line_number} is not valid JSON: {exc}') from exc

EXPECTED_KEYS = {
    'dataset_schema_version', 'source_schema_version', 'task_type', 'split',
    'provenance', 'group_hash', 'input_hash', 'example_hash', 'messages',
}
HEX64 = re.compile(r'^[a-f0-9]{64}$')
PRIVACY_PATTERNS = {
    'email': re.compile(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b'),
    'clerk_user_id': re.compile(r'\buser_[A-Za-z0-9]{6,}\b'),
    'absolute_path': re.compile(r'(?i)(?:/(?:Users|home|var|tmp)/[^\s]+|[A-Z]:\\[^\s]+)'),
    'credential': re.compile(r'(?i)\b(?:bearer\s+[A-Za-z0-9._-]{8,}|sk-[A-Za-z0-9_-]{8,}|api[_-]?(?:key|token)\s*[:=]\s*[^\s]+)'),
    'filename': re.compile(r'(?i)\b[^\s/\\]+\.(?:csv|tsv|xls|xlsx|doc|docx|pdf|json|zip)\b'),
}

assert len(rows) >= MIN_EXAMPLES, f'Need at least {MIN_EXAMPLES} exportable examples; found {len(rows)}.'
group_splits, input_splits = defaultdict(set), defaultdict(set)
task_counts, split_counts = Counter(), Counter()
split_groups = defaultdict(set)
example_hashes, input_hashes = set(), set()
privacy_findings = []

for index, row in enumerate(rows, start=1):
    assert isinstance(row, dict) and set(row) == EXPECTED_KEYS, f'Row {index} has an unexpected schema.'
    assert row['dataset_schema_version'] == DATASET_SCHEMA_VERSION, f'Row {index} uses an unsupported dataset schema.'
    assert isinstance(row['source_schema_version'], int) and row['source_schema_version'] >= 1
    assert row['task_type'] in REQUIRED_TASKS, f'Row {index} has an unknown task.'
    assert row['split'] in {'train', 'validation', 'test'}, f'Row {index} has an invalid split.'
    assert row['provenance'] in ALLOWED_PROVENANCE, f'Row {index} has provenance incompatible with {TRAINING_DATA_MODE}.'
    for key in ('group_hash', 'input_hash', 'example_hash'):
        assert isinstance(row[key], str) and HEX64.fullmatch(row[key]), f'Row {index} has an invalid {key}.'
    assert row['example_hash'] not in example_hashes, f'Duplicate example at row {index}.'
    assert row['input_hash'] not in input_hashes, f'Duplicate or conflicting input at row {index}.'
    example_hashes.add(row['example_hash']); input_hashes.add(row['input_hash'])

    messages = row['messages']
    assert isinstance(messages, list) and len(messages) >= 3, f'Row {index} has too few messages.'
    assert messages[0].get('role') == 'system', f'Row {index} lacks its system message.'
    assert messages[-1].get('role') == 'assistant', f'Row {index} lacks its target completion.'
    assert all(isinstance(message, dict) and set(message) == {'role', 'content'} for message in messages)
    assert all(message['role'] in {'system', 'user', 'assistant'} for message in messages)
    assert all(isinstance(message['content'], str) and message['content'].strip() for message in messages)
    assert not any(message['role'] == 'system' for message in messages[1:])
    assert any(message['role'] == 'user' for message in messages), f'Row {index} has no user message.'
    assert messages[-2]['role'] == 'user', f'Row {index} does not end with a user prompt before the correction.'
    expected_roles = ['user' if turn % 2 == 0 else 'assistant' for turn in range(len(messages) - 1)]
    assert [message['role'] for message in messages[1:]] == expected_roles, f'Row {index} roles do not alternate.'
    assert f"<TASK={row['task_type']}>" in messages[0]['content'], f'Row {index} has a mismatched task tag.'

    joined = '\n'.join(message['content'] for message in messages)
    for finding, pattern in PRIVACY_PATTERNS.items():
        if pattern.search(joined):
            privacy_findings.append({'row': index, 'example_hash': row['example_hash'], 'finding': finding})

    task_counts[row['task_type']] += 1
    split_counts[row['split']] += 1
    split_groups[row['split']].add(row['group_hash'])
    group_splits[row['group_hash']].add(row['split'])
    input_splits[row['input_hash']].add(row['split'])

assert not privacy_findings, f'Privacy scan failed. Quarantine report: {privacy_findings[:10]}'
assert all(len(splits) == 1 for splits in group_splits.values()), 'A project/experiment group leaked across splits.'
assert all(len(splits) == 1 for splits in input_splits.values()), 'An identical input leaked across splits.'
missing_tasks = REQUIRED_TASKS - set(task_counts)
underrepresented = {task: task_counts[task] for task in REQUIRED_TASKS if task_counts[task] < MIN_PER_TASK}
assert not missing_tasks, f'Missing task coverage: {sorted(missing_tasks)}'
assert not underrepresented, f'Need at least {MIN_PER_TASK} examples per task: {underrepresented}'
assert split_counts['train'] > 0, 'The training split is empty.'
for split in ('validation', 'test'):
    assert split_counts[split] >= MIN_HOLDOUT_EXAMPLES, f'{split} needs at least {MIN_HOLDOUT_EXAMPLES} examples.'
    assert len(split_groups[split]) >= MIN_HOLDOUT_GROUPS, f'{split} needs at least {MIN_HOLDOUT_GROUPS} independent groups.'
    heldout_tasks = {row['task_type'] for row in rows if row['split'] == split}
    assert heldout_tasks == REQUIRED_TASKS, f'{split} is missing task coverage: {sorted(REQUIRED_TASKS - heldout_tasks)}'

RUN_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{dataset_sha256[:12]}"
RUN_ROOT = RUNS_ROOT / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
if bootstrap_manifest_path is not None:
    bootstrap_manifest = json.loads(bootstrap_manifest_path.read_text())
    assert bootstrap_manifest['dataset_sha256'] == dataset_sha256, 'Bootstrap manifest does not match the dataset.'
    (RUN_ROOT / 'public-bootstrap-source-manifest.json').write_text(json.dumps(bootstrap_manifest, indent=2) + '\n')
DATASET_SUMMARY = {
    'dataset_sha256': dataset_sha256, 'dataset_schema_version': DATASET_SCHEMA_VERSION,
    'training_data_mode': TRAINING_DATA_MODE,
    'provenance_counts': dict(Counter(row['provenance'] for row in rows)),
    'examples': len(rows), 'task_counts': dict(sorted(task_counts.items())),
    'split_counts': dict(split_counts),
    'split_group_counts': {split: len(groups) for split, groups in split_groups.items()},
    'privacy_scan_findings': 0,
}
(RUN_ROOT / 'dataset-validation.json').write_text(json.dumps(DATASET_SUMMARY, indent=2) + '\n')
print(json.dumps(DATASET_SUMMARY, indent=2))
print(f'Validated dataset fingerprint: {dataset_sha256}')

## 2. Prove that no training example will be truncated

Scientific instructions and target answers must fit completely inside the 2,048-token training window. Over-length examples stop the run instead of silently chopping off the answer.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = 'right'

token_lengths, completion_lengths, overlong = [], [], []
prepared = {'train': [], 'validation': [], 'test': []}
for row in rows:
    full_text = tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)
    full_ids = tokenizer(full_text, add_special_tokens=False)['input_ids']
    completion_ids = tokenizer(row['messages'][-1]['content'], add_special_tokens=False)['input_ids']
    length, completion_length = len(full_ids), len(completion_ids)
    token_lengths.append(length); completion_lengths.append(completion_length)
    if length > MAX_LENGTH:
        overlong.append({'example_hash': row['example_hash'], 'task_type': row['task_type'], 'tokens': length})
    assert completion_length >= 4, f"Completion is too short to teach from: {row['example_hash']}"
    prepared[row['split']].append({
        'prompt': row['messages'][:-1],
        'completion': [row['messages'][-1]],
    })

TOKEN_AUDIT = {
    'max_allowed': MAX_LENGTH, 'minimum': min(token_lengths), 'maximum': max(token_lengths),
    'mean': statistics.fmean(token_lengths),
    'p95': float(np.percentile(token_lengths, 95)),
    'completion_minimum': min(completion_lengths),
    'overlong_count': len(overlong), 'overlong_examples': overlong,
}
(RUN_ROOT / 'token-audit.json').write_text(json.dumps(TOKEN_AUDIT, indent=2) + '\n')
assert not overlong, (
    f'{len(overlong)} examples exceed {MAX_LENGTH} tokens. Nothing was truncated. '
    f'Review token-audit.json; fix the public source selection or shorten the human-reviewed website examples, then rebuild.'
)
dataset = DatasetDict({split: Dataset.from_list(items) for split, items in prepared.items()})
print(json.dumps(TOKEN_AUDIT, indent=2))
dataset

## 3. Load the pinned base model in 4-bit and train completion-only

Only the source-backed assistant completion contributes to loss. System instructions and user context are inputs, not labels. Checkpoints go to the private Drive folder so a Colab interruption can resume.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainerCallback

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, quantization_config=quantization,
    device_map={'': 0}, attn_implementation='sdpa',
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

lora = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM', target_modules=['q_proj', 'v_proj'],
)

class FiniteMetricsCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        for key in ('loss', 'eval_loss', 'grad_norm'):
            value = (logs or {}).get(key)
            if value is not None and not math.isfinite(float(value)):
                raise FloatingPointError(f'Non-finite {key} at step {state.global_step}: {value}')

print(f'4-bit base loaded; LoRA compute dtype: {compute_dtype}')

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = RUN_ROOT / 'checkpoints'
effective_batch_size = 16
steps_per_epoch = max(1, math.ceil(len(dataset['train']) / effective_batch_size))
eval_steps = max(5, min(25, steps_per_epoch))
TRAINING_CONFIG = {
    'base_model': BASE_MODEL, 'base_revision': BASE_REVISION,
    'lora_rank': 8, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'v_proj'], 'max_length': MAX_LENGTH,
    'epochs': 2, 'learning_rate': 2e-4, 'effective_batch_size': effective_batch_size,
    'seed': SEED, 'optimizer': 'paged_adamw_8bit', 'completion_only_loss': True,
}
config = SFTConfig(
    output_dir=str(OUTPUT_DIR), max_length=MAX_LENGTH, truncation_mode='keep_start',
    num_train_epochs=2, per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine',
    optim='paged_adamw_8bit', weight_decay=0.01, max_grad_norm=0.3,
    fp16=compute_dtype == torch.float16, bf16=compute_dtype == torch.bfloat16,
    logging_steps=5, eval_strategy='steps', eval_steps=eval_steps,
    save_strategy='steps', save_steps=eval_steps, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    report_to='none', run_name=RUN_ID, seed=SEED, data_seed=SEED,
    push_to_hub=False, packing=False, eval_packing=False, completion_only_loss=True, shuffle_dataset=True,
    remove_unused_columns=True, save_safetensors=True,
)
trainer = SFTTrainer(
    model=model, args=config, peft_config=lora, processing_class=tokenizer,
    train_dataset=dataset['train'], eval_dataset=dataset['validation'],
    callbacks=[FiniteMetricsCallback()],
)
trainer.model.print_trainable_parameters()
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
assert 0 < trainable / total < 0.01, 'Unexpected trainable-parameter ratio; refusing to train.'

checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*'), key=lambda path: int(path.name.split('-')[-1]))
resume_checkpoint = str(checkpoints[-1]) if checkpoints else None
print(f'Resuming from {resume_checkpoint}' if resume_checkpoint else 'Starting a new adapter run.')
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
validation_metrics = trainer.evaluate(dataset['validation'], metric_key_prefix='validation')
assert math.isfinite(float(train_result.training_loss)), 'Training loss is not finite.'
assert math.isfinite(float(validation_metrics['validation_loss'])), 'Validation loss is not finite.'
(RUN_ROOT / 'trainer-log-history.json').write_text(json.dumps(trainer.state.log_history, indent=2) + '\n')
print({'train_loss': train_result.training_loss, **validation_metrics})

## 4. Verify and stage a Cloudflare-compatible adapter

This saves only LoRA tensors, rejects full-model files or quantized adapter tensors, enforces Cloudflare's rank/size/filename rules, and writes an audit manifest without prompts or responses.

In [ ]:
from peft import PeftConfig
from safetensors import safe_open

ADAPTER_DIR = RUN_ROOT / 'verified-adapter'
ADAPTER_DIR.mkdir(parents=True, exist_ok=False)
trainer.model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(ADAPTER_DIR / 'tokenizer-audit-only'))

config_path = ADAPTER_DIR / 'adapter_config.json'
weights_path = ADAPTER_DIR / 'adapter_model.safetensors'
assert config_path.is_file() and weights_path.is_file(), 'Required adapter files are missing.'
adapter_config = json.loads(config_path.read_text())
adapter_config['model_type'] = 'mistral'
config_path.write_text(json.dumps(adapter_config, indent=2, sort_keys=True) + '\n')

reloaded_config = PeftConfig.from_pretrained(str(ADAPTER_DIR))
assert reloaded_config.r == 8, f'Cloudflare adapter rank must be 8; got {reloaded_config.r}.'
assert set(reloaded_config.target_modules) == {'q_proj', 'v_proj'}
assert adapter_config['model_type'] == 'mistral'
assert weights_path.stat().st_size < 300 * 1024 * 1024, 'Adapter exceeds Cloudflare 300 MB limit.'
assert not list(ADAPTER_DIR.glob('model*.safetensors')), 'A full model file was accidentally saved.'
assert not list(ADAPTER_DIR.glob('pytorch_model*.bin')), 'A full model file was accidentally saved.'

with safe_open(str(weights_path), framework='pt', device='cpu') as adapter_file:
    tensor_keys = list(adapter_file.keys())
    assert tensor_keys and all('lora_' in key for key in tensor_keys), 'Unexpected non-LoRA tensors found.'
    tensor_dtypes = {str(adapter_file.get_tensor(key).dtype) for key in tensor_keys}
assert tensor_dtypes <= {'torch.float16', 'torch.bfloat16', 'torch.float32'}, f'Quantized adapter tensors found: {tensor_dtypes}'

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

TRAINING_MANIFEST = {
    'run_id': RUN_ID, 'created_at': datetime.now(timezone.utc).isoformat(),
    'dataset': DATASET_SUMMARY, 'token_audit': {k: v for k, v in TOKEN_AUDIT.items() if k != 'overlong_examples'},
    'training_config': TRAINING_CONFIG, 'package_versions': PACKAGE_VERSIONS,
    'gpu': {'name': gpu.name, 'memory_gib': gpu_gib},
    'metrics': {'train_loss': train_result.training_loss, **validation_metrics},
    'adapter': {
        'rank': reloaded_config.r, 'target_modules': sorted(reloaded_config.target_modules),
        'tensor_dtypes': sorted(tensor_dtypes), 'tensor_count': len(tensor_keys),
        'adapter_config_sha256': sha256_file(config_path),
        'adapter_model_sha256': sha256_file(weights_path),
        'adapter_model_bytes': weights_path.stat().st_size,
    },
}
(ADAPTER_DIR / 'training_manifest.json').write_text(json.dumps(TRAINING_MANIFEST, indent=2) + '\n')
print(json.dumps(TRAINING_MANIFEST['adapter'], indent=2))

## 5. Generate a blind base-vs-adapter review

Every held-out test prompt is run through both the pinned base model and the adapter with deterministic decoding. Candidate order is blinded. Automated JSON and numeric checks are flags—not substitutes for the scientist review.

In [ ]:
NUMBER_RE = re.compile(r'(?<![A-Za-z])[-+]?\d+(?:,\d{3})*(?:\.\d+)?(?:[eE][-+]?\d+)?%?')
DERIVED_WORDS = ('derived', 'calculated', 'computed', 'estimated', 'ratio', 'mean', 'average', 'sd', 'cv')

def unsupported_numeric_claims(answer, prompt):
    prompt_numbers = {match.group(0).replace(',', '') for match in NUMBER_RE.finditer(prompt)}
    unsupported = []
    for match in NUMBER_RE.finditer(answer):
        value = match.group(0).replace(',', '')
        nearby = answer[max(0, match.start() - 80):match.start()].lower()
        if value not in prompt_numbers and not any(word in nearby for word in DERIVED_WORDS):
            unsupported.append(value)
    return sorted(set(unsupported))

def parse_json_if_expected(text, expected):
    if not expected:
        return True
    try:
        json.loads(text.strip().removeprefix('```json').removesuffix('```').strip())
        return True
    except Exception:
        return False

def generate_for_prompt(prompt_messages, completion_text, adapter_enabled):
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors='pt', add_special_tokens=False).to('cuda')
    reference_tokens = len(tokenizer(completion_text, add_special_tokens=False)['input_ids'])
    max_new_tokens = min(768, max(128, math.ceil(reference_tokens * 1.25)))
    context = contextlib.nullcontext() if adapter_enabled else trainer.model.disable_adapter()
    with context, torch.inference_mode():
        generated = trainer.model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

trainer.model.config.use_cache = True
trainer.model.eval()
test_rows = [row for row in rows if row['split'] == 'test']
evaluation_records, blind_key = [], {}
for position, row in enumerate(test_rows, start=1):
    prompt_messages, reference = row['messages'][:-1], row['messages'][-1]['content']
    prompt_text = '\n'.join(message['content'] for message in prompt_messages)
    base_answer = generate_for_prompt(prompt_messages, reference, adapter_enabled=False)
    adapter_answer = generate_for_prompt(prompt_messages, reference, adapter_enabled=True)
    structured = reference.lstrip().startswith(('{', '['))
    swap = int(row['example_hash'][:8], 16) % 2 == 1
    candidates = {'a': adapter_answer if swap else base_answer, 'b': base_answer if swap else adapter_answer}
    mapping = {'a': 'adapter' if swap else 'base', 'b': 'base' if swap else 'adapter'}
    example_prefix = row['example_hash'][:12]
    evaluation_id = f'eval-{position:04d}-{example_prefix}'
    blind_key[evaluation_id] = mapping
    record = {
        'evaluation_id': evaluation_id, 'task_type': row['task_type'],
        'input_context': prompt_text, 'reference_output': reference,
        'reference_is_structured': structured,
        'candidate_a': candidates['a'], 'candidate_b': candidates['b'],
        'candidate_a_json_valid': parse_json_if_expected(candidates['a'], structured),
        'candidate_b_json_valid': parse_json_if_expected(candidates['b'], structured),
        'candidate_a_numeric_flags': json.dumps(unsupported_numeric_claims(candidates['a'], prompt_text)),
        'candidate_b_numeric_flags': json.dumps(unsupported_numeric_claims(candidates['b'], prompt_text)),
        'candidate_a_rating': '', 'candidate_b_rating': '',
        'candidate_a_approved': '', 'candidate_b_approved': '',
        'candidate_a_schema_valid': '', 'candidate_b_schema_valid': '',
        'candidate_a_measurement_fidelity': '', 'candidate_b_measurement_fidelity': '',
        'candidate_a_privacy_pass': '', 'candidate_b_privacy_pass': '',
        'reviewer_notes': '',
    }
    evaluation_records.append(record)
    print(f'Generated {position}/{len(test_rows)} held-out comparisons')

REVIEW_CSV = RUN_ROOT / 'biolab-blind-review.csv'
with REVIEW_CSV.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(evaluation_records[0]))
    writer.writeheader(); writer.writerows(evaluation_records)
(RUN_ROOT / 'blind-key.json').write_text(json.dumps(blind_key, indent=2) + '\n')
files.download(str(REVIEW_CSV))
print('Fill every review field without opening blind-key.json. Use true/false for all pass/approval fields and ratings 1–5.')

## 6. Human scientific review and release gates

Review both candidates without opening `blind-key.json`. For structured tasks, validate against the website's current response schema. For measurements, confirm quoted values exist in the input and any new math is explicitly described as derived. Mark privacy pass only when no identity or cross-user content appears. Save as `biolab-blind-review-completed.csv`, then upload it below.

In [ ]:
review_upload = files.upload()
review_names = [name for name in review_upload if name.endswith('.csv')]
assert len(review_names) == 1, 'Upload exactly one completed review CSV.'
review_text = review_upload[review_names[0]].decode('utf-8')
reviewed = list(csv.DictReader(review_text.splitlines()))
assert len(reviewed) == len(evaluation_records), 'The completed review must contain every held-out row exactly once.'
assert {row['evaluation_id'] for row in reviewed} == set(blind_key), 'Evaluation IDs changed or are missing.'

def required_bool(value, field, evaluation_id):
    normalized = str(value).strip().lower()
    assert normalized in {'true', 'false'}, f'{evaluation_id}: {field} must be true or false.'
    return normalized == 'true'

scores = {'adapter': [], 'base': []}
approvals = {'adapter': [], 'base': []}
schema_passes, fidelity_passes, privacy_passes = {'adapter': [], 'base': []}, {'adapter': [], 'base': []}, {'adapter': [], 'base': []}
json_passes = {'adapter': [], 'base': []}
record_by_id = {record['evaluation_id']: record for record in evaluation_records}

for row in reviewed:
    evaluation_id = row['evaluation_id']
    original = record_by_id[evaluation_id]
    for candidate in ('a', 'b'):
        model_kind = blind_key[evaluation_id][candidate]
        try:
            rating = int(row[f'candidate_{candidate}_rating'])
        except Exception as exc:
            raise AssertionError(f'{evaluation_id}: candidate {candidate.upper()} needs a rating from 1 to 5.') from exc
        assert 1 <= rating <= 5
        scores[model_kind].append(rating)
        approvals[model_kind].append(required_bool(row[f'candidate_{candidate}_approved'], 'approved', evaluation_id))
        fidelity_passes[model_kind].append(required_bool(row[f'candidate_{candidate}_measurement_fidelity'], 'measurement_fidelity', evaluation_id))
        privacy_passes[model_kind].append(required_bool(row[f'candidate_{candidate}_privacy_pass'], 'privacy_pass', evaluation_id))
        if original['reference_is_structured']:
            schema_passes[model_kind].append(required_bool(row[f'candidate_{candidate}_schema_valid'], 'schema_valid', evaluation_id))
            json_passes[model_kind].append(bool(original[f'candidate_{candidate}_json_valid']))

adapter_high_rating_rate = sum(score >= 4 for score in scores['adapter']) / len(scores['adapter'])
adapter_approval_rate = sum(approvals['adapter']) / len(approvals['adapter'])
base_approval_rate = sum(approvals['base']) / len(approvals['base'])
GATES = {
    'structured_responses_100_percent_valid': all(json_passes['adapter']) and all(schema_passes['adapter']),
    'measurements_100_percent_faithful': all(fidelity_passes['adapter']),
    'privacy_100_percent_pass': all(privacy_passes['adapter']),
    'adapter_answers_rated_4_or_5_at_least_80_percent': adapter_high_rating_rate >= 0.80,
    'adapter_approval_improves_by_15_points': adapter_approval_rate - base_approval_rate >= 0.15,
}
RELEASE_GATES_PASSED = all(GATES.values())
RELEASE_REPORT = {
    'run_id': RUN_ID, 'dataset_sha256': dataset_sha256, 'reviewed_examples': len(reviewed),
    'adapter_high_rating_rate': adapter_high_rating_rate,
    'adapter_approval_rate': adapter_approval_rate, 'base_approval_rate': base_approval_rate,
    'approval_improvement_points': adapter_approval_rate - base_approval_rate,
    'gates': GATES, 'release_gates_passed': RELEASE_GATES_PASSED,
}
(RUN_ROOT / 'release-gate-report.json').write_text(json.dumps(RELEASE_REPORT, indent=2) + '\n')
(RUN_ROOT / 'biolab-blind-review-completed.csv').write_text(review_text)
print(json.dumps(RELEASE_REPORT, indent=2))
assert RELEASE_GATES_PASSED, 'Adapter is quarantined. Keep CLOUDFLARE_LORA_ID empty, collect corrections, and retrain.'

## 7. Publish only the accepted adapter to a private Hugging Face repository

This cell cannot run unless every release gate passed. It uploads only the small adapter plus non-sensitive audit artifacts. The full Mistral model and the training JSONL are never uploaded.

In [ ]:
assert RELEASE_GATES_PASSED, 'Release gates did not pass; publishing is blocked.'
notebook_login()  # use a write token; never paste it into a notebook cell
HF_USER = whoami()['name']
HUB_MODEL_ID = (f'{HF_USER}/biolab-ai-mistral-lora-public-bootstrap'
                if TRAINING_DATA_MODE == 'public_bootstrap'
                else f'{HF_USER}/biolab-ai-mistral-lora')
print(f'Private adapter destination: {HUB_MODEL_ID}')
TRAINING_MANIFEST['release_evaluation'] = RELEASE_REPORT
(ADAPTER_DIR / 'training_manifest.json').write_text(json.dumps(TRAINING_MANIFEST, indent=2) + '\n')
(ADAPTER_DIR / 'release-gate-report.json').write_text(json.dumps(RELEASE_REPORT, indent=2) + '\n')
(ADAPTER_DIR / 'README.md').write_text(
    '# Private Bio-Lab AI LoRA adapter\n\n'
    f'Base: `{BASE_MODEL}` at `{BASE_REVISION}`.\n\n'
    f'Dataset fingerprint: `{dataset_sha256}`. No dataset rows are stored in this repository.\n'
)

api = HfApi()
api.create_repo(repo_id=HUB_MODEL_ID, repo_type='model', private=True, exist_ok=True)
repo = api.repo_info(repo_id=HUB_MODEL_ID, repo_type='model')
assert repo.private is True, 'Refusing to upload: the Hugging Face repository is not private.'
api.upload_folder(
    repo_id=HUB_MODEL_ID, repo_type='model', folder_path=str(ADAPTER_DIR),
    ignore_patterns=['tokenizer-audit-only/**'],
)
print(f'Accepted private adapter: https://huggingface.co/{HUB_MODEL_ID}')
print('Cloudflare upload must use only adapter_config.json and adapter_model.safetensors. Keep CLOUDFLARE_LORA_ID empty until that upload succeeds and production smoke tests pass.')